# kikoyu — 会議音声の文字起こし＋話者分離 (DGX Spark / Blackwell sm_121 版)

会議などの音声ファイルを **DGX Spark (GB10 / Blackwell sm_121 / ARM64 / CUDA 13)** ローカルで
文字起こしし、`pyannote` で話者分離するノートブックです。録音や文字起こし結果、固有名詞辞書は
リポジトリに含めません (`.gitignore` 済み)。詳しくは `README.md` を参照してください。

> **前提: このノートブックは DGX Spark / Blackwell 専用です。** 後述のパッチ群とコンテナは
> sm_121 + CUDA 13 という特殊な組み合わせを動かすためのもので、一般的な x86 + CUDA 12 環境では
> そのままでは使えません (その環境なら素の WhisperX がそのまま動きます)。

## DGX Spark 環境前提・既知の落とし穴

このノートブックは 2026-05 時点の DGX Spark 環境 (aarch64 + CUDA 13) で
**実際に whisper-large-v3 文字起こし + pyannote 話者分離が成功した構成**を再現用に整理したものです。
以下の落とし穴を踏まえて作られています:

| # | 制約 / 罠 | 本ノートブックでの対応 |
| --- | --- | --- |
| 1 | DGX Spark は NVRTC が sm_121 未対応で素の `pip install whisperx` では GPU で動かない | [Mekopa/whisperx-blackwell](https://github.com/Mekopa/whisperx-blackwell) コンテナ (PyTorch capability spoof で sm_121→sm_90/H100 偽装) を使用 |
| 2 | CTranslate2 4.4.0 の aarch64+CUDA13 wheel が CUDA13非互換 → **faster-whisper が起動できない** | 文字起こしは `transformers.pipeline` + `bf16` + `chunk_length_s=30` で実施 (60分音声で実測 ~60分)。faster-whisper セルは「参考・休止中」として残置 |
| 3 | PyTorch 2.6+ `weights_only=True` デフォルト化で pyannote / Silero VAD のロードが失敗 | 冒頭の統合パッチセルで `torch.load` を `weights_only=False` に上書き |
| 4 | `pyannote/speaker-diarization-3.1` のチェックポイントに含まれる `TorchVersion` が SemVer 不正で失敗 | `torch.__version__ = "2.6.0"` で NGC の `a0+...` サフィックスを剥がし、`add_safe_globals([TorchVersion])` 併用 |
| 5 | `lightning_fabric.utilities.cloud_io._load` も内部で `weights_only=True` を渡してくる | 同じく冒頭で monkey-patch |
| 6 | VS Code Jupyter 拡張が XSRF チェックで Jupyter Lab を弾く | Docker の `jupyter lab` 起動時に `--ServerApp.disable_check_xsrf=True` を必ず付ける |
| 7 | HF token をノートブックにハードコードすると共有時に流出 | `.env` (chmod 600) に書いて `--env-file .env` で渡し、ノートブック内では `os.environ["HF_whisperx"]` のみ参照 |

## 動かし方

### 1. DGX Spark 側でコンテナを起動

DGX Spark に SSH している前提です (接続先ホストは各自の環境に置き換えてください)。
作業ディレクトリに、このノートブック (`kikoyu.ipynb`) と音声ファイル (`*.m4a` 等) を置きます。

```bash
# DGX Spark 上で
mkdir -p ~/kikoyu-work && cd ~/kikoyu-work
# このリポジトリ (kikoyu.ipynb, prompts/ 等) と音声ファイルをここに置く

docker pull mekopa/whisperx-blackwell:latest

# pyannote 話者分離用の HF token は .env に置く (初回のみ・以後は不要):
#   cd ~/kikoyu-work && printf 'HF_whisperx=hf_xxxxxxxxxxxx\n' > .env && chmod 600 .env
#   ※ 値はクオートで囲まない (--env-file はクオートも値の一部として扱うため)

docker run -it --rm \
  --gpus all --ipc=host --ulimit memlock=-1 \
  -p 8888:8888 \
  -v "$PWD:/work" -w /work \
  --env-file .env \
  --entrypoint bash \
  mekopa/whisperx-blackwell:latest \
  -lc "pip install -q jupyterlab ipywidgets && \
       jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root \
                   --ServerApp.token='' --ServerApp.password='' \
                   --ServerApp.disable_check_xsrf=True \
                   --ServerApp.root_dir=/work"
```

> `--ServerApp.disable_check_xsrf=True` は VS Code Jupyter 拡張から接続するときに必須。
> ブラウザだけで使うなら不要。

### 2. 手元のマシンから Jupyter を開く

別ターミナルから SSH ポートフォワードでトンネリングし、ブラウザで `http://localhost:8888/lab` を開きます。

```bash
ssh -L 8888:localhost:8888 <あなたの DGX Spark への SSH 先>
```

接続が切れやすい環境では、SSH 先で `tmux new -s kikoyu` を実行してから `docker run` すると
セッションを保護できます (`tmux attach -t kikoyu` で復帰)。
VS Code から直接接続する場合は Jupyter 拡張の "Connect to existing Jupyter server" で
`http://localhost:8888` を指定します。

### 3. このノートブックを開いて上から順に実行

- **必ず Cell 1 (PyTorch パッチ) から実行する**。これを飛ばすと話者分離が落ちます。
- 文字起こしを既に終えていて話者分離だけやり直したい場合は、
  Cell 1 (パッチ) → Cell 3 (環境チェック) → Cell 5 (パス) → **`## 6. 保存済み segments から再開する場合`** → 話者分離セル の順で進めてください。


## 0. PyTorch パッチ (最初に必ず実行)

DGX Spark + NGC PyTorch 25.01 + pyannote 3.1 の組み合わせで必要な monkey-patch を
**ノートブックの先頭で 1 回だけ**実行します。
カーネル再起動後はこのセルから走らせ直してください。

- `torch.load` を `weights_only=False` で固定 (PyTorch 2.6+ デフォルト変更対応)
- `torch.__version__` を `"2.6.0"` に偽装 (NGC `a0+...` SemVer 不正による pyannote の TorchVersion 解釈失敗を回避)
- `add_safe_globals([TorchVersion])` で pyannote チェックポイント内の TorchVersion を許可
- `lightning_fabric.utilities.cloud_io._load` を `weights_only=False` で上書き


In [ ]:
# === DGX Spark / pyannote 互換パッチ (二重適用ガードつき) ============================
# 詳細は冒頭の "DGX Spark 環境前提・既知の落とし穴" を参照。
import torch
from torch.serialization import add_safe_globals
from torch.torch_version import TorchVersion

# A. torch.load を weights_only=False で覆う (Silero VAD / pyannote の両方が依存)
if not getattr(torch.load, "_wx_patched", False):
    _orig_torch_load = torch.load
    def _torch_load_compat(*args, **kwargs):
        kwargs["weights_only"] = False
        return _orig_torch_load(*args, **kwargs)
    _torch_load_compat._wx_patched = True
    torch.load = _torch_load_compat

# B. pyannote のチェックポイントに含まれる TorchVersion を許可
add_safe_globals([TorchVersion])

# C. lightning_fabric の _load も置換 (こちらが weights_only=True を内側で渡してくる)
import lightning_fabric.utilities.cloud_io as _cio
if not getattr(_cio._load, "_wx_patched", False):
    _orig_cio_load = _cio._load
    def _patched_cio_load(path_or_url, map_location, weights_only=False):
        return _orig_cio_load(path_or_url, map_location, weights_only=False)
    _patched_cio_load._wx_patched = True
    _cio._load = _patched_cio_load

# D. torch.__version__ を SemVer 適合形に偽装 (NGC の "2.6.0a0+..." を pyannote が解釈できない)
torch.__version__ = "2.6.0"

print(f"torch.load patched : {getattr(torch.load, '_wx_patched', False)}")
print(f"cio._load patched  : {getattr(_cio._load, '_wx_patched', False)}")
print(f"torch.__version__  : {torch.__version__}")


## 1. 環境チェック

`get_device_capability()` が `(9, 0)` を返すのが Mekopa の spoof が効いているサインです
(物理的には sm_121 ですが、Hopper / H100 に偽装することで sm_90 用カーネルにフォールバックします)。


In [ ]:
import os, sys, platform
import torch

print("python      :", sys.version.split()[0])
print("platform    :", platform.platform())
print("torch       :", torch.__version__)
print("torch CUDA  :", torch.version.cuda)
print("cuDNN       :", torch.backends.cudnn.version())
print("CUDA avail  :", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU が見えていません。--gpus all 付きで起動していますか?"
print("GPU         :", torch.cuda.get_device_name(0))
print("capability  :", torch.cuda.get_device_capability(0), "<- (9, 0) なら spoof OK")
print("arch list   :", torch.cuda.get_arch_list())

import whisperx, ctranslate2, faster_whisper
print("whisperx    :", whisperx.__version__ if hasattr(whisperx, "__version__") else "(no __version__)")
print("ctranslate2 :", ctranslate2.__version__)
print("faster-whisper:", faster_whisper.__version__)


## 2. 入出力パス

`/work` はホスト側 `~/whisperx-work` をマウントしています。

**音声ファイルは冒頭の `AUDIO_FILE` で明示指定します（明示が最優先）。**
- `AUDIO_FILE = "..."` に処理したいファイル名を書けば、必ずそれを使います（毎月の本番はこれ）。
  `/work` 相対のファイル名でも絶対パスでも構いません。
- `AUDIO_FILE = ""`（空文字）のときだけ `/work` 直下を自動検出にフォールバックします。
  ただし**候補がちょうど 1 件のときだけ**採用し、0 件・2 件以上なら候補を一覧表示して
  例外で停止します（黙って最新を選ばないので、フォルダに複数あっても取り違えに必ず気づけます）。
- コンテナ起動時に `-e AUDIO=xxx.m4a` を渡すと、その場限りで `AUDIO_FILE` より優先されます。

出力は `{stem}_segments.json` (生 segments) と `{stem}_transcript.txt` (タイムスタンプつき素テキスト)、
話者分離後は `{stem}_segments_speaker.json` / `{stem}_transcript_speaker.txt` です。

In [ ]:
import os
from pathlib import Path

WORK    = Path("/work")
out_dir = WORK / "out"
out_dir.mkdir(parents=True, exist_ok=True)

# === 音声ファイルの指定 ===============================================
# 設計方針: 「明示指定が最優先・自動検出は曖昧なら止める」。
#   1. AUDIO_FILE に処理したいファイル名を書けば、必ずそれを使う (本番はこれ)。
#      /work 相対のファイル名でも絶対パスでも可。
#   2. AUDIO_FILE が空文字 "" のときだけ /work 直下を自動検出にフォールバックする。
#   3. 自動検出は候補がちょうど 1 件のときだけ採用する。
#      0 件 or 2 件以上なら候補を一覧表示して例外で停止する
#      (黙って「最新」を選ばない。取り違えに必ず気づける形にする)。
#   なお環境変数 AUDIO があれば、コンテナ起動時の一時上書きとして AUDIO_FILE より優先する。
# ----------------------------------------------------------------------
AUDIO_FILE = ""   # ← 処理するファイル名を書く。例: "2026-06-15-meeting.m4a"。"" なら /work 直下を自動検出
# ----------------------------------------------------------------------

AUDIO_EXTS = {".m4a", ".mp3", ".wav", ".mp4", ".aac", ".flac"}

_explicit = os.environ.get("AUDIO", "").strip() or AUDIO_FILE.strip()

if _explicit:
    p = Path(_explicit)
    audio_path = str(p if p.is_absolute() else WORK / _explicit)
    if not Path(audio_path).exists():
        raise FileNotFoundError(f"明示指定された音声ファイルがありません: {audio_path}")
    print("音声 (明示指定):", audio_path)
else:
    cands = sorted(
        p for p in WORK.iterdir()
        if p.is_file() and p.suffix.lower() in AUDIO_EXTS
    )
    if len(cands) != 1:
        listing = "\n".join(f"    - {p.name}" for p in cands) if cands else "    (なし)"
        raise RuntimeError(
            f"自動検出で音声ファイルを一意に決められません ({len(cands)} 件)。\n"
            f"  候補:\n{listing}\n"
            f"  → ノートブック冒頭の AUDIO_FILE に処理したいファイル名を明示してください。"
        )
    audio_path = str(cands[0])
    print("音声 (自動検出):", audio_path, "  (候補1件のみ)")

stem = Path(audio_path).stem
print("出力先         :", out_dir, f"(stem={stem})")

## 3. `initial_prompt` (固有名詞辞書)

役員名・地名・専門用語などを「、」区切りで列挙すると認識精度が上がります。
**この辞書は実名などの個人情報を含みうるため、リポジトリには含めません。**

- 本番用の辞書は `prompts/initial_prompt.local.txt` に置きます (`.gitignore` 済み・git に乗りません)。
- `local.txt` が無ければ汎用サンプル `prompts/initial_prompt.example.txt` を使い、警告を出します。
- どちらも無ければ `initial_prompt` 無しで続行します。

`transformers.pipeline` 経由では現状 `language="ja"` 強制のため prompt は間接的にしか効きませんが、
faster-whisper パス (後述・休止中) では明示的に利用されます。


In [ ]:
# 固有名詞辞書 (initial_prompt) を外部ファイルから読み込みます。
# 個人情報 (実名など) を含みうるため、本体は prompts/initial_prompt.local.txt に置き、
# git にはコミットしません (.gitignore 済み)。local が無ければ汎用サンプルを使い、警告します。
from pathlib import Path

_prompt_dir = Path("/work/prompts")
_local   = _prompt_dir / "initial_prompt.local.txt"
_example = _prompt_dir / "initial_prompt.example.txt"

if _local.exists():
    initial_prompt = _local.read_text(encoding="utf-8").strip()
    _src = _local.name
elif _example.exists():
    initial_prompt = _example.read_text(encoding="utf-8").strip()
    _src = _example.name
    print("\u26a0\ufe0f  prompts/initial_prompt.local.txt が無いため汎用サンプルを使用します。")
    print("    精度を上げるには、固有名詞辞書を prompts/initial_prompt.local.txt に用意してください。")
else:
    initial_prompt = ""
    _src = "none"
    print("\u26a0\ufe0f  prompts/ に initial_prompt が見つかりません。initial_prompt 無しで続行します。")

print(f"initial_prompt: {len(initial_prompt)} 文字  (source: {_src})")


## 4. 文字起こし (transformers — DGX Spark 現状デフォルト)

DGX Spark (aarch64 + CUDA 13) では CTranslate2 4.4.0 の wheel が CUDA13非互換のため
**faster-whisper が起動できません**。
そのため現状のデフォルトは HuggingFace `transformers.pipeline` 経由 + `bf16` です。
60 分の音声で実測 ~60 分 (RTF ≈ 1.0)。

- `chunk_length_s=30`: Whisper のネイティブ窓に合わせる
- `batch_size=4`: GB10 (128GB unified) でも `=8` で OOM したため 4 に固定
- `no_repeat_ngram_size=3` + 多段 temperature: 繰り返し loop の抑止
- 出力 `segments` は `[{"start": float, "end": float, "text": str}, ...]` の形

faster-whisper 版は後段の「## 4'. (参考・休止中) faster-whisper 版」セルに残してあります。
CTranslate2 が CUDA 13 対応 wheel をリリースしたらそちらに戻す予定です。


In [ ]:
# transformers 経由の場合は librosa で 16kHz mono に揃えてから渡すと安定する
import os, librosa
audio_array, sr = librosa.load(audio_path, sr=16000, mono=True)
print(f"loaded: {len(audio_array)/sr/60:.1f} min, sr={sr}")

In [ ]:
# transformers.pipeline で whisper-large-v3 を bf16 で動かす (DGX Spark 現状デフォルト)
# 60分音声で実測 ~60分。batch_size=8 だと OOM するので 4 固定。
import torch
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
    device="cuda",
    torch_dtype=torch.bfloat16,
)

result_raw = pipe(
    {"raw": audio_array, "sampling_rate": sr},
    return_timestamps=True,
    chunk_length_s=30,
    batch_size=4,
    generate_kwargs={
        "language": "ja",
        "task": "transcribe",
        "no_repeat_ngram_size": 3,                # 繰り返し抑止
        "compression_ratio_threshold": 1.35,
        "logprob_threshold": -1.0,
        "no_speech_threshold": 0.6,
        "temperature": (0.0, 0.2, 0.4, 0.6, 0.8), # fallback温度
        "max_new_tokens": 256,
    },
)

segments = [
    {"start": c["timestamp"][0], "end": c["timestamp"][1], "text": c["text"]}
    for c in result_raw["chunks"]
]
# 後段 (アラインメント / 話者分離) が期待する形にも揃えておく
result = {"segments": segments, "language": "ja"}

print(f"segments: {len(segments)}")
for seg in segments[:20]:
    print(f"[{seg['start']:7.1f}s → {seg['end']:7.1f}s] {seg['text']}")
if len(segments) > 20:
    print(f"... ({len(segments)-20} segments omitted)")


## 4'. (参考・休止中) faster-whisper 版

> **現状このセルは実行できません。**
> CTranslate2 4.4.0 の aarch64 wheel が CUDA 13 非互換 (NVRTC エラーで落ちる) のため、
> faster-whisper はロード時点で失敗します。CT2 が CUDA 13 対応 wheel をリリースしたら、
> 上の transformers 版に代わってこちらをデフォルトに戻します。

faster-whisper パスは VAD 自動チャンク化が効くため、RTF が transformers より速くなる見込みです。


In [ ]:
# ⚠️ DGX Spark (CUDA 13) では現状動作しません。CT2 が CUDA 13 対応するまで休止中。
# --- PyTorch 2.6+ weights_only デフォルト化対策 (このセルだけ単独実行する場合のガード) ---
import torch
if not getattr(torch.load, "_wx_patched", False):
    _orig_torch_load = torch.load
    def _torch_load_compat(*args, **kwargs):
        kwargs["weights_only"] = False
        return _orig_torch_load(*args, **kwargs)
    _torch_load_compat._wx_patched = True
    torch.load = _torch_load_compat
# --- ここまで ---

import gc, time
from faster_whisper import WhisperModel

device       = "cuda"
compute_type = "float16"
beam_size    = 5

model = WhisperModel("large-v3", device=device, compute_type=compute_type)

t0 = time.perf_counter()
segments_iter, info = model.transcribe(
    audio_path,
    language="ja",
    initial_prompt=initial_prompt,
    beam_size=beam_size,
    vad_filter=True,
    vad_parameters=dict(min_silence_duration_ms=500),
)

segments = []
for s in segments_iter:
    segments.append({"start": s.start, "end": s.end, "text": s.text})

elapsed = time.perf_counter() - t0
dur_sec = info.duration
print(f"audio duration : {dur_sec/60:.1f} min")
print(f"detected lang  : {info.language} (prob {info.language_probability:.2f})")
print(f"transcribe     : {elapsed:.1f}s  (RTF={elapsed/dur_sec:.3f}x)")

result = {"segments": segments, "language": info.language}

del model
torch.cuda.empty_cache()
gc.collect()

print(f"\nsegments: {len(segments)}\n")
for seg in segments[:20]:
    print(f"[{seg['start']:7.1f}s → {seg['end']:7.1f}s] {seg['text']}")
if len(segments) > 20:
    print(f"... ({len(segments)-20} segments omitted)")


## 5. 保存

文字起こし結果を 2 ファイルで書き出します:

- `{stem}_segments.json` — `segments` リストをそのまま (後段の再開・アラインメント・話者分離の入力)
- `{stem}_transcript.txt` — `[ssss.ss] text` 形式のプレーンテキスト


In [ ]:
import json, pathlib

out_dir = pathlib.Path("/work/out")
out_dir.mkdir(parents=True, exist_ok=True)
stem = pathlib.Path(audio_path).stem

seg_path = out_dir / f"{stem}_segments.json"
txt_path = out_dir / f"{stem}_transcript.txt"

seg_path.write_text(json.dumps(segments, ensure_ascii=False, indent=2), encoding="utf-8")
with open(txt_path, "w", encoding="utf-8") as f:
    for seg in segments:
        f.write(f"[{seg['start']:7.1f}s] {seg['text']}\n")

print("written:")
for p in (seg_path, txt_path):
    print(" ", p, f"({p.stat().st_size:,} bytes)")


## 6. 保存済み segments から再開する場合

文字起こしは時間がかかる (60分音声で ~60分) ので、JSON で保存しておけば
カーネル再起動後に文字起こしをスキップして直接アラインメント / 話者分離に進めます。

使い方:

1. カーネル再起動
2. `## 0. PyTorch パッチ` (Cell 1) を実行
3. `## 1. 環境チェック` (Cell 3) を実行
4. `## 2. 入出力パス` (Cell 5) を実行
5. **このセル** を実行 (文字起こしセクションは飛ばす)
6. 必要なら `## 7. アラインメント` / `## 8. 話者分離` へ


In [ ]:
# 保存済み segments を再ロードして、文字起こしセルと同じ状態に復元する
import json, pathlib

# audio_path は「2. 入出力パス」セルで自動検出済みの想定。
# 単独で動かして未定義なら、先にそのセルを実行するよう促す。
try:
    audio_path
except NameError as e:
    raise RuntimeError("先に『2. 入出力パス』セルを実行してください (audio_path 未定義)") from e
stem = pathlib.Path(audio_path).stem

segments = json.loads(
    (pathlib.Path("/work/out") / f"{stem}_segments.json").read_text(encoding="utf-8")
)

# 後段のアラインメント / 話者分離が期待する形に揃える
result = {"segments": segments, "language": "ja"}

print(f"reloaded segments: {len(segments)}")
print(f"first segment    : [{segments[0]['start']:.1f}s] {segments[0]['text'][:60]}")
print(f"last segment     : [{segments[-1]['start']:.1f}s] {segments[-1]['text'][:60]}")


## 7. (任意) 単語単位タイムスタンプのアラインメント

WhisperX の本領。`large-v3` のセグメント単位タイムスタンプを wav2vec2 系モデルで再アラインして、
単語単位のタイムスタンプを取り直します。日本語は `jonatasgrosman/wav2vec2-large-xlsr-53-japanese` 等が使えます。

話者分離だけなら必須ではない (`RUN_ALIGN = False` のままで OK)。


In [ ]:
# 必要なときだけ実行
RUN_ALIGN = False

if RUN_ALIGN:
    import gc, json, pathlib, torch, whisperx
    device = "cuda" if torch.cuda.is_available() else "cpu"
    audio = whisperx.load_audio(audio_path)

    align_model, metadata = whisperx.load_align_model(
        language_code="ja",
        device=device,
        model_name="jonatasgrosman/wav2vec2-large-xlsr-53-japanese",
    )
    result_aligned = whisperx.align(
        result["segments"], align_model, metadata, audio, device,
        return_char_alignments=False,
    )
    del align_model
    torch.cuda.empty_cache()
    gc.collect()

    out_json_aligned = pathlib.Path("/work/out") / f"{stem}_segments_aligned.json"
    with open(out_json_aligned, "w", encoding="utf-8") as f:
        json.dump(result_aligned, f, ensure_ascii=False, indent=2)
    print("aligned ->", out_json_aligned)
else:
    print("skip (RUN_ALIGN=False)")


## 8. (任意) 話者分離 (pyannote)

会議参加者が複数いる場合に、誰が何を言ったかを分けたいときに使います。
[HuggingFace](https://huggingface.co/pyannote/speaker-diarization-3.1) で
`pyannote/speaker-diarization-3.1` と `pyannote/segmentation-3.0` の利用規約に同意して
HF token を作り、**`.env` に書いて `--env-file .env` で渡しておきます**
(ノートブックにはハードコードしない)。

出力:

- `{stem}_segments_speaker.json` — 話者ラベル付き segments (JSON)
- `{stem}_transcript_speaker.txt` — `[HH:MM:SS] SPEAKER_XX: text` 形式のテキスト


In [ ]:
import os, torch, whisperx

device = "cuda" if torch.cuda.is_available() else "cpu"
assert os.environ.get("HF_whisperx"), \
    "HF_whisperx が未設定です。.env に書いて --env-file .env で渡してください"

audio = whisperx.load_audio(audio_path)
print(f"device={device}, audio shape={audio.shape}, audio_path={audio_path}")

In [ ]:
RUN_DIARIZE = True

if RUN_DIARIZE:
    import os, gc, json, pathlib, torch, whisperx
    hf_token = os.environ["HF_whisperx"]

    # whisperx 3.8 系は DiarizationPipeline の場所が変わっているので両対応
    try:
        from whisperx.diarize import DiarizationPipeline
    except ImportError:
        from whisperx import DiarizationPipeline

    diarize_model = DiarizationPipeline(use_auth_token=hf_token, device=device)
    # 参加者数に応じて min/max を調整 (不明なら未指定でも可)
    diarize_segments = diarize_model(audio, min_speakers=2, max_speakers=30)

    # align 済 result があればそれを使う、なければ素の result
    base = result_aligned if ("RUN_ALIGN" in dir() and RUN_ALIGN) else result
    result_speakers = whisperx.assign_word_speakers(diarize_segments, base)

    del diarize_model
    torch.cuda.empty_cache()
    gc.collect()

    def fmt_hms(sec: float) -> str:
        h = int(sec // 3600); m = int((sec % 3600) // 60); s = int(sec % 60)
        return f"{h:02d}:{m:02d}:{s:02d}"

    out_dir = pathlib.Path("/work/out")
    stem = pathlib.Path(audio_path).stem
    spk_json = out_dir / f"{stem}_segments_speaker.json"
    spk_txt  = out_dir / f"{stem}_transcript_speaker.txt"

    # JSON: result_speakers["segments"] をそのまま保存
    spk_json.write_text(
        json.dumps(result_speakers["segments"], ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    # TXT: [HH:MM:SS] SPEAKER_XX: text
    with open(spk_txt, "w", encoding="utf-8") as f:
        for seg in result_speakers["segments"]:
            spk = seg.get("speaker", "UNK")
            f.write(f"[{fmt_hms(seg['start'])}] {spk}: {seg['text'].strip()}\n")

    print("written:")
    for p in (spk_json, spk_txt):
        print(" ", p, f"({p.stat().st_size:,} bytes)")
else:
    print("skip (RUN_DIARIZE=False)")


## 9. トラブルシュート

| 症状 | 対処 |
| --- | --- |
| `torch.cuda.is_available() == False` | コンテナを `--gpus all` 付きで起動したか確認。`nvidia-smi` が DGX Spark ホスト側で通ることが前提 |
| `get_device_capability() == (12, 1)` | Mekopa の spoof が当たっていない。`mekopa/whisperx-blackwell:latest` を使っているか再確認 |
| `kernel ... is for sm80-sm100, but was built for sm121` | flash-attn / xformers を入れ直していないか確認。コンテナ素のままを使う |
| `ValueError: ... not compiled with CUDA support` (ctranslate2) | PyPI の aarch64 wheel が混ざっている。`pip show ctranslate2` で Mekopa 同梱版 (source build) になっているか確認。**現状 CT2 4.4.0 は CUDA13非互換のため faster-whisper パスは動かないのが正常** — transformers パスを使う |
| `UnpicklingError ... weights_only` / `add_safe_globals` 関連 | Cell 1 (PyTorch パッチ) を実行していない可能性。カーネル再起動後は必ずパッチセルから走らせる |
| `Invalid version: 'X.Y.Za0+...'` (pyannote ロード時) | `torch.__version__` 偽装が効いていない。Cell 1 を実行したか確認 |
| VS Code Jupyter 拡張から接続できない (XSRF エラー) | Docker 起動コマンドに `--ServerApp.disable_check_xsrf=True` を付ける |
| OOM (transformers パス) | `batch_size=4` を `=2` に下げる、`torch_dtype=torch.float16` に変える |
| 漢字が化ける | 出力ファイルは UTF-8 で書いています。Windows のメモ帳ではなく VSCode などで開いてください |

### 参考
- Mekopa/whisperx-blackwell : <https://github.com/Mekopa/whisperx-blackwell>
- WhisperX issue #1326 (DGX Spark で動かす) : <https://github.com/m-bain/whisperX/issues/1326>
- NVIDIA forum (sm_121 サポート) : <https://forums.developer.nvidia.com/t/dgx-spark-sm121-software-support-is-severely-lacking-official-roadmap-needed/357663>
